In [1]:
from sklearn.neighbors import KNeighborsClassifier
from scipy.io import arff
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, KFold
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder 
import numpy as np
from sklearn.metrics import accuracy_score

In [62]:
#First, I'm going to create a list of lists of every possible parameter value that I can use in the main algorithm:

parameters = [
        [5, 'uniform', 'minkowski'],
        [5, 'uniform', 'cityblock'],
        [5, 'uniform', 'cosine'],
        [5, 'distance', 'minkowski'],
        [5, 'distance', 'cityblock'],
        [5, 'distance', 'cosine'],
        [6, 'uniform', 'minkowski'],
        [6, 'uniform', 'cityblock'],
        [6, 'uniform', 'cosine'],
        [6, 'distance', 'minkowski'],
        [6, 'distance', 'cityblock'],
        [6, 'distance', 'cosine'],
        [7, 'uniform', 'minkowski'],
        [7, 'uniform', 'cityblock'],
        [7, 'uniform', 'cosine'],
        [7, 'distance', 'minkowski'],
        [7, 'distance', 'cityblock'],
        [7, 'distance', 'cosine']]

#Second, I need to prepare my datasets again so they can be used:

#Sick:

pd.set_option('display.max_rows', 30)

# Load ARFF file
arff_file_path = 'sick_numeric2.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data) 

Features = []
Classes = []
for i in range(3772):
    row_as_list = df.iloc[i].tolist()
    Classes.append(row_as_list[29])
    del row_as_list[29]
    Features.append(row_as_list)    

X, y = Features, Classes
#
X = np.array(X)
#convirt from bit strings to numeric values, where 1 is positive and -1 is negetive 
y = [-1 if cls == b'negative' else 1 for cls in y]
y = np.array(y)

final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    #sets the training and testing data
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        #Further breaks up the training data in 2/3 training and 1/3 validation
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #loops through every possble combination of parameters using Minkowski, cosine, and cityblock as metrics
        Accuracy_list = []
        for params in parameters:
            knn = KNeighborsClassifier(n_neighbors=params[0], weights=params[1], metric=params[2])
            knn.fit(X_train_smaller, y_train_smaller)
            y_pred_val = knn.predict(X_val)
            accuracy = accuracy_score(y_val, y_pred_val)
            Accuracy_list.append(accuracy)
        #finds the list of parameters that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_parameters = parameters[max_accuracy_index]
        #creates a new model that's using the best parameters and evaluates it using the entire training split and the test split
        knn_best = KNeighborsClassifier(n_neighbors=best_parameters[0], weights=best_parameters[1], metric=best_parameters[2])
        knn_best.fit(X_train, y_train)
        y_pred_test = knn_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    #finds the mean accuracy across all folds of this iteration of r
    mean_accuracy_all_folds = sum(final_accuracies) / len(final_accuracies)
    #adds this accuracy to a list
    final_accuracies0.append(mean_accuracy_all_folds)
#calculates a final mean accuracy and standard deviation across all values of r and all folds
mean_accuracies_all_r = sum(final_accuracies0) / len(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)



Mean Accuracy: 0.9490954766816836
Standard Deviation: 0.0


In [20]:
#letter:

pd.set_option('display.max_rows', 30)

# Load ARFF file
arff_file_path = 'dataset_6_letter.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data)
#in this Pandas dataframe, the column lables are each of the features (16 in total), the final column is the class of letter, and everything 
#in the other columns is the feature value for each sample, ir observation. In order to utalize a pandas datafram, I need to create a 
#2D array (list of list) consisting of a list of every feature value in each sample: [[ft1,ft2,ft3...],[ft1,ft2,ft3...],...]
#This 2d array will be X. 
#y will be the class lable that correspods to each of the samples, or lists of features. y will be a array in the form [class1, class2, class3, ...].
#I now need to disect the data from the pandas dataframe. 

Features = []
Classes = []
for i in range(20000):
    row_as_list = df.iloc[i].tolist()
    Classes.append(row_as_list[16])
    del row_as_list[16]
    Features.append(row_as_list)    

X, y = Features, Classes

X = np.array(X)
# Convert bit strings to integers (1 for 'A', 2 for 'B', ..., 26 for 'Z')
y = [ord(cls) - 64 for cls in y]
y = np.array(y)

final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    #sets the training and testing data
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        #Further breaks up the training data in 2/3 training and 1/3 validation
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #loops through every possble combination of parameters using Minkowski, cosine, and cityblock as metrics
        Accuracy_list = []
        for params in parameters:
            knn = KNeighborsClassifier(n_neighbors=params[0], weights=params[1], metric=params[2])
            knn.fit(X_train_smaller, y_train_smaller)
            y_pred_val = knn.predict(X_val)
            accuracy = accuracy_score(y_val, y_pred_val)
            Accuracy_list.append(accuracy)
        #finds the list of parameters that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_parameters = parameters[max_accuracy_index]
        #creates a new model that's using the best parameters and evaluates it using the entire training split and the test split
        knn_best = KNeighborsClassifier(n_neighbors=best_parameters[0], weights=best_parameters[1], metric=best_parameters[2])
        knn_best.fit(X_train, y_train)
        y_pred_test = knn_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    #finds the mean accuracy across all folds of this iteration of r
    mean_accuracy_all_folds = sum(final_accuracies) / len(final_accuracies)
    #adds this accuracy to a list
    final_accuracies0.append(mean_accuracy_all_folds)
#calculates a final mean accuracy and standard deviation across all values of r and all folds
mean_accuracies_all_r = sum(final_accuracies0) / len(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)



Mean Accuracy: 0.9593999999999999
Standard Deviation: 0.0


In [34]:
#Ionosphere

# Load ARFF file
arff_file_path = 'dataset_59_ionosphere.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data)

Features = []
Classes = []
for i in range(351):
    row_as_list = df.iloc[i].tolist()
    Classes.append(row_as_list[34])
    del row_as_list[34]
    Features.append(row_as_list)    

X, y = Features, Classes

X = np.array(X)
# Convert bit strings to integers 1 and -1
y = [-1 if cls == b'b' else 1 for cls in y]
y = np.array(y)

final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    #sets the training and testing data
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        #Further breaks up the training data in 2/3 training and 1/3 validation
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #loops through every possble combination of parameters using Minkowski, cosine, and cityblock as metrics
        Accuracy_list = []
        for params in parameters:
            knn = KNeighborsClassifier(n_neighbors=params[0], weights=params[1], metric=params[2])
            knn.fit(X_train_smaller, y_train_smaller)
            y_pred_val = knn.predict(X_val)
            accuracy = accuracy_score(y_val, y_pred_val)
            Accuracy_list.append(accuracy)
        #finds the list of parameters that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_parameters = parameters[max_accuracy_index]
        #creates a new model that's using the best parameters and evaluates it using the entire training split and the test split
        knn_best = KNeighborsClassifier(n_neighbors=best_parameters[0], weights=best_parameters[1], metric=best_parameters[2])
        knn_best.fit(X_train, y_train)
        y_pred_test = knn_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    #finds the mean accuracy across all folds of this iteration of r
    mean_accuracy_all_folds = sum(final_accuracies) / len(final_accuracies)
    #adds this accuracy to a list
    final_accuracies0.append(mean_accuracy_all_folds)
#calculates a final mean accuracy and standard deviation across all values of r and all folds
mean_accuracies_all_r = sum(final_accuracies0) / len(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)


Mean Accuracy: 0.8861111111111111
Standard Deviation: 0.0


In [63]:
from scipy.io import arff
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder 
import numpy as np
pd.set_option('display.max_rows', 30)

# Load ARFF file
arff_file_path = 'solar.flare2.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data)

#This solar flair data has 1066 samples a d 13 attributes (including the 3 classes to predict). All samples are from one active reigon of the sun.
#The attributes are:
# %  Attribute Information:
# %    1. Code for class (modified Zurich class)  (A,B,C,D,E,F,H) 
# %    2. Code for largest spot size              (X,R,S,A,H,K) 
# %    3. Code for spot distribution              (X,O,I,C) 
# %    4. Activity                                (1 = reduced, 2 = unchanged)
# %    5. Evolution                               (1 = decay, 2 = no growth, 
# %                                                3 = growth)
# %    6. Previous 24 hour flare activity code    (1 = nothing as big as an M1,
# %                                                2 = one M1,
# %                                                3 = more activity than one M1)
# %    7. Historically-complex                    (1 = Yes, 2 = No)
# %    8. Did region become historically complex  (1 = yes, 2 = no) 
# %       on this pass across the sun's disk
# %    9. Area                                    (1 = small, 2 = large)
# %   10. Area of the largest spot                (1 = <=5, 2 = >5)
# % 
# %  From all these predictors three classes of flares are predicted, which are 
# %  represented in the last three columns.
# % 
# %   11. C-class flares production by this region    Number  
# %       in the following 24 hours (common flares)
# %   12. M-class flares production by this region    Number
# %       in the following 24 hours (moderate flares)
# %   13. X-class flares production by this region    Number
# %       in the following 24 hours (severe flares)

#The three classes to predict (C-class, M-class, X-class) output numbers that indicate how many of those classes of flares were detected
#in the active riegon that is being viewed. So, it is possible for a signle sample of data to have numbers in all three classes. 

df['class'] = df['class'].astype('category') 
df['largest_spot_size'] = df['largest_spot_size'].astype('category')
df['spot_distribution'] = df['spot_distribution'].astype('category') 

df['new_class'] = df['class'].cat.codes
df['new_largest_spot_size'] = df['largest_spot_size'].cat.codes
df['new_spot_distribution'] = df['spot_distribution'].cat.codes

X_train_categorical = df[['class','largest_spot_size','spot_distribution']]

encoder = OneHotEncoder()
X_train_encoded = encoder.fit_transform(X_train_categorical)  # Encode categorical features

# Convert the encoded data to a DataFrame
X_train_encoded_df = pd.DataFrame(X_train_encoded.toarray(), columns=encoder.get_feature_names_out())

# Remove original categorical columns from the original DataFrame
df = df.drop(columns=X_train_categorical.columns)

# Concatenate the one-hot encoded DataFrame with the remaining columns in df
df = pd.concat([df, X_train_encoded_df], axis=1)

Features = []
Classes = []


for i in range(1066):
    row_as_list = df.iloc[i].tolist()
    Classes.append([row_as_list[26],row_as_list[27],row_as_list[28]])
    del row_as_list[28]
    del row_as_list[27]
    del row_as_list[26]
    # Append the remaining elements of row_as_list to Features
    Features.append(row_as_list)

X, y = Features, Classes


#convirt bit strings to integers
#Note: be careful when doing this for k-nearest neighbor.
#If the original features were listed as letters to mean different types,
#changing them to numbers will tell the algorithm that some are closer to the others numerically.
#changing them to numeric type is not the same thing as One-hot encoading.
#Now that the first three columns have been one-hot encoded, I can change their bit string values to numeric ones. 

for j in range(1066):
    X[j] = [0 if cls == b'0' else
         1 if cls == b'1' else 
         2 if cls == b'2' else 
         3 if cls == b'3' else 
         4 if cls == b'4' else 
         5 if cls == b'5' else 
         6 if cls == b'6' else 
         7 if cls == b'7' else cls 
         for cls in X[j]]

#if the encoded bit strings are not turned into integer type before the data is converted to a Numpy array,
#all of the data that was integer type turns into bir strings. 
X = np.array(X, dtype=int)

y = np.array(y)

final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    #sets the training and testing data
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        #Further breaks up the training data in 2/3 training and 1/3 validation
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #loops through every possble combination of parameters using Minkowski, cosine, and cityblock as metrics
        Accuracy_list = []
        for params in parameters:
            knn = KNeighborsClassifier(n_neighbors=params[0], weights=params[1], metric=params[2])
            knn.fit(X_train_smaller, y_train_smaller)
            y_pred_val = knn.predict(X_val)
            accuracy = accuracy_score(y_val, y_pred_val)
            Accuracy_list.append(accuracy)
        #finds the list of parameters that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_parameters = parameters[max_accuracy_index]
        #creates a new model that's using the best parameters and evaluates it using the entire training split and the test split
        knn_best = KNeighborsClassifier(n_neighbors=best_parameters[0], weights=best_parameters[1], metric=best_parameters[2])
        knn_best.fit(X_train, y_train)
        y_pred_test = knn_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    #finds the mean accuracy across all folds of this iteration of r
    mean_accuracy_all_folds = sum(final_accuracies) / len(final_accuracies)
    #adds this accuracy to a list
    final_accuracies0.append(mean_accuracy_all_folds)
#calculates a final mean accuracy and standard deviation across all values of r and all folds
mean_accuracies_all_r = sum(final_accuracies0) / len(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)


Mean Accuracy: 0.9540557220948684
Standard Deviation: 2.220446049250313e-16


In [64]:
#Decision tree version:

In [2]:
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

# Replace the original parameters list with alpha values for post-pruning
#I need to figure out which values for alpha are the best to use. When alpha is just set to zero, the decision tree overfits.
#The larger alpha becomes, the more the tree is pruned.   

#Sick:

pd.set_option('display.max_rows', 30)

# Load ARFF file
arff_file_path = 'sick_numeric2.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data) 

Features = []
Classes = []
for i in range(3772):
    row_as_list = df.iloc[i].tolist()
    Classes.append(row_as_list[29])
    del row_as_list[29]
    Features.append(row_as_list)    

X, y = Features, Classes

X = np.array(X)
#convirt from bit strings to numeric values, where 1 is positive and -1 is negetive 
y = [-1 if cls == b'negative' else 1 for cls in y]
y = np.array(y)


final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #determine the best values for alpha quickly before trying each on the validation set. 
        tree = DecisionTreeClassifier(random_state=0)
        path = tree.cost_complexity_pruning_path(X_train_smaller, y_train_smaller)
        ccp_alphas, impurities = path.ccp_alphas, path.impurities
        
        #The ccp_alphas always contains zero. Zero will always end up being the best alpha value too because it causes
        #the tree to overfit. For that reason, I'll throw out zero
        ccp_alphas = ccp_alphas[ccp_alphas != 0]
        
        Accuracy_list = []
        for ccp_alpha in ccp_alphas[:-1]:
            #set to :-1 so the entire tree doesnt get pruned
            clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
            clf.fit(X_train_smaller, y_train_smaller)
    
            # Evaluate on the training set
            y_train_pred = clf.predict(X_train_smaller)
            train_accuracy = accuracy_score(y_train_smaller, y_train_pred)
            Accuracy_list.append(train_accuracy)
        
        # Find the alpha value that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_alpha = ccp_alphas[max_accuracy_index]
        # Create a new model using the best alpha value and test it on the test set
        dtree_best = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
        dtree_best.fit(X_train, y_train)
        y_pred_test = dtree_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    
    mean_accuracy_all_folds = np.mean(final_accuracies)
    final_accuracies0.append(mean_accuracy_all_folds)

mean_accuracies_all_r = np.mean(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)


Mean Accuracy: 0.9822365374089511
Standard Deviation: 0.0


In [11]:
#letter:

pd.set_option('display.max_rows', 30)

# Load ARFF file
arff_file_path = 'dataset_6_letter.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data)
#in this Pandas dataframe, the column lables are each of the features (16 in total), the final column is the class of letter, and everything 
#in the other columns is the feature value for each sample, ir observation. In order to utalize a pandas datafram, I need to create a 
#2D array (list of list) consisting of a list of every feature value in each sample: [[ft1,ft2,ft3...],[ft1,ft2,ft3...],...]
#This 2d array will be X. 
#y will be the class lable that correspods to each of the samples, or lists of features. y will be a array in the form [class1, class2, class3, ...].
#I now need to disect the data from the pandas dataframe. 

Features = []
Classes = []
for i in range(20000):
    row_as_list = df.iloc[i].tolist()
    Classes.append(row_as_list[16])
    del row_as_list[16]
    Features.append(row_as_list)    

X, y = Features, Classes

X = np.array(X)
# Convert bit strings to integers (1 for 'A', 2 for 'B', ..., 26 for 'Z')
y = [ord(cls) - 64 for cls in y]
y = np.array(y)
#for some reason after the loop is printed out 10 times, the mean and standard deviationare still not printed out
#and the code continues to run. Any idea why the code is being halted towards the end?  
final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #determine the best values for alpha quickly before trying each on the validation set. 
        tree = DecisionTreeClassifier(random_state=0)
        path = tree.cost_complexity_pruning_path(X_train_smaller, y_train_smaller)
        ccp_alphas, impurities = path.ccp_alphas, path.impurities
        
        #The ccp_alphas always contains zero. Zero will always end up being the best alpha value too because it causes
        #the tree to overfit. For that reason, I'll throw out zero
        ccp_alphas = ccp_alphas[ccp_alphas != 0]
        Accuracy_list = []
        for ccp_alpha in ccp_alphas[:-1]:
            #set to :-1 so the entire tree doesnt get pruned
            #build a model on each
            clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
            clf.fit(X_train_smaller, y_train_smaller)
            # Evaluate on the training set
            y_train_pred = clf.predict(X_train_smaller)
            train_accuracy = accuracy_score(y_train_smaller, y_train_pred)
            Accuracy_list.append(train_accuracy)
        # Find the alpha value that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_alpha = ccp_alphas[max_accuracy_index]
        # Create a new model using the best alpha value and test it on the test set
        dtree_best = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
        dtree_best.fit(X_train, y_train)
        y_pred_test = dtree_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    mean_accuracy_all_folds = np.mean(final_accuracies)
    final_accuracies0.append(mean_accuracy_all_folds)

mean_accuracies_all_r = np.mean(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)

Mean Accuracy: 0.8788
Standard Deviation: 0.0


In [81]:
#Ionosphere

# Load ARFF file
arff_file_path = 'dataset_59_ionosphere.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data)

Features = []
Classes = []
for i in range(351):
    row_as_list = df.iloc[i].tolist()
    Classes.append(row_as_list[34])
    del row_as_list[34]
    Features.append(row_as_list)    

X, y = Features, Classes

X = np.array(X)
# Convert bit strings to integers 1 and -1
y = [-1 if cls == b'b' else 1 for cls in y]
y = np.array(y)


final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #determine the best values for alpha quickly before trying each on the validation set. 
        tree = DecisionTreeClassifier(random_state=0)
        path = tree.cost_complexity_pruning_path(X_train_smaller, y_train_smaller)
        ccp_alphas, impurities = path.ccp_alphas, path.impurities
        
        #The ccp_alphas always contains zero. Zero will always end up being the best alpha value too because it causes
        #the tree to overfit. For that reason, I'll throw out zero
        ccp_alphas = ccp_alphas[ccp_alphas != 0]
        Accuracy_list = []
        for ccp_alpha in ccp_alphas[:-1]:
            #set to :-1 so the entire tree doesnt get pruned
            clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
            clf.fit(X_train_smaller, y_train_smaller)

            # Evaluate on the training set
            y_train_pred = clf.predict(X_train_smaller)
            train_accuracy = accuracy_score(y_train_smaller, y_train_pred)
            Accuracy_list.append(train_accuracy)
        # Find the alpha value that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_alpha = ccp_alphas[max_accuracy_index]
        # Create a new model using the best alpha value and test it on the test set
        dtree_best = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
        dtree_best.fit(X_train, y_train)
        y_pred_test = dtree_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    mean_accuracy_all_folds = np.mean(final_accuracies)
    final_accuracies0.append(mean_accuracy_all_folds)

mean_accuracies_all_r = np.mean(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)

Mean Accuracy: 0.8832539682539682
Standard Deviation: 0.0


In [82]:
pd.set_option('display.max_rows', 30)

# Load ARFF file
arff_file_path = 'solar.flare2.arff'  
data, meta = arff.loadarff(arff_file_path)

# Convert data to a Pandas DataFrame
df = pd.DataFrame(data)

#This solar flair data has 1066 samples a d 13 attributes (including the 3 classes to predict). All samples are from one active reigon of the sun.
#The attributes are:
# %  Attribute Information:
# %    1. Code for class (modified Zurich class)  (A,B,C,D,E,F,H) 
# %    2. Code for largest spot size              (X,R,S,A,H,K) 
# %    3. Code for spot distribution              (X,O,I,C) 
# %    4. Activity                                (1 = reduced, 2 = unchanged)
# %    5. Evolution                               (1 = decay, 2 = no growth, 
# %                                                3 = growth)
# %    6. Previous 24 hour flare activity code    (1 = nothing as big as an M1,
# %                                                2 = one M1,
# %                                                3 = more activity than one M1)
# %    7. Historically-complex                    (1 = Yes, 2 = No)
# %    8. Did region become historically complex  (1 = yes, 2 = no) 
# %       on this pass across the sun's disk
# %    9. Area                                    (1 = small, 2 = large)
# %   10. Area of the largest spot                (1 = <=5, 2 = >5)
# % 
# %  From all these predictors three classes of flares are predicted, which are 
# %  represented in the last three columns.
# % 
# %   11. C-class flares production by this region    Number  
# %       in the following 24 hours (common flares)
# %   12. M-class flares production by this region    Number
# %       in the following 24 hours (moderate flares)
# %   13. X-class flares production by this region    Number
# %       in the following 24 hours (severe flares)

#The three classes to predict (C-class, M-class, X-class) output numbers that indicate how many of those classes of flares were detected
#in the active riegon that is being viewed. So, it is possible for a signle sample of data to have numbers in all three classes. 

df['class'] = df['class'].astype('category') 
df['largest_spot_size'] = df['largest_spot_size'].astype('category')
df['spot_distribution'] = df['spot_distribution'].astype('category') 

df['new_class'] = df['class'].cat.codes
df['new_largest_spot_size'] = df['largest_spot_size'].cat.codes
df['new_spot_distribution'] = df['spot_distribution'].cat.codes

X_train_categorical = df[['class','largest_spot_size','spot_distribution']]

encoder = OneHotEncoder()
X_train_encoded = encoder.fit_transform(X_train_categorical)  # Encode categorical features

# Convert the encoded data to a DataFrame
X_train_encoded_df = pd.DataFrame(X_train_encoded.toarray(), columns=encoder.get_feature_names_out())

# Remove original categorical columns from the original DataFrame
df = df.drop(columns=X_train_categorical.columns)

# Concatenate the one-hot encoded DataFrame with the remaining columns in df
df = pd.concat([df, X_train_encoded_df], axis=1)

Features = []
Classes = []


for i in range(1066):
    row_as_list = df.iloc[i].tolist()
    Classes.append([row_as_list[26],row_as_list[27],row_as_list[28]])
    del row_as_list[28]
    del row_as_list[27]
    del row_as_list[26]
    # Append the remaining elements of row_as_list to Features
    Features.append(row_as_list)

X, y = Features, Classes


#convirt bit strings to integers
#Note: be careful when doing this for k-nearest neighbor.
#If the original features were listed as letters to mean different types,
#changing them to numbers will tell the algorithm that some are closer to the others numerically.
#changing them to numeric type is not the same thing as One-hot encoading.
#Now that the first three columns have been one-hot encoded, I can change their bit string values to numeric ones. 

for j in range(1066):
    X[j] = [0 if cls == b'0' else
         1 if cls == b'1' else 
         2 if cls == b'2' else 
         3 if cls == b'3' else 
         4 if cls == b'4' else 
         5 if cls == b'5' else 
         6 if cls == b'6' else 
         7 if cls == b'7' else cls 
         for cls in X[j]]

#if the encoded bit strings are not turned into integer type before the data is converted to a Numpy array,
#all of the data that was integer type turns into bir strings. 
X = np.array(X, dtype=int)

y = np.array(y)

final_accuracies0 = []
for r in range(10):  # R iterations
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    final_accuracies = []
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        X_train_smaller, X_val, y_train_smaller, y_val = train_test_split(
            X_train, y_train, test_size=1/3, random_state=42)
        #determine the best values for alpha quickly before trying each on the validation set. 
        tree = DecisionTreeClassifier(random_state=0)
        path = tree.cost_complexity_pruning_path(X_train_smaller, y_train_smaller)
        ccp_alphas, impurities = path.ccp_alphas, path.impurities
        
        #The ccp_alphas always contains zero. Zero will always end up being the best alpha value too because it causes
        #the tree to overfit. For that reason, I'll throw out zero
        ccp_alphas = ccp_alphas[ccp_alphas != 0]
        Accuracy_list = []
        for ccp_alpha in ccp_alphas[:-1]:
            #set to :-1 so the entire tree doesnt get pruned
            clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
            clf.fit(X_train_smaller, y_train_smaller)

            # Evaluate on the training set
            y_train_pred = clf.predict(X_train_smaller)
            train_accuracy = accuracy_score(y_train_smaller, y_train_pred)
            Accuracy_list.append(train_accuracy)
        # Find the alpha value that had the highest accuracy
        max_accuracy_index = Accuracy_list.index(max(Accuracy_list))
        best_alpha = ccp_alphas[max_accuracy_index]
        # Create a new model using the best alpha value and test it on the test set
        dtree_best = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
        dtree_best.fit(X_train, y_train)
        y_pred_test = dtree_best.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        final_accuracies.append(accuracy_test)
    mean_accuracy_all_folds = np.mean(final_accuracies)
    final_accuracies0.append(mean_accuracy_all_folds)

mean_accuracies_all_r = np.mean(final_accuracies0)
std_dev = np.std(final_accuracies0)

print("Mean Accuracy:", mean_accuracies_all_r)
print("Standard Deviation:", std_dev)

Mean Accuracy: 0.9962616822429908
Standard Deviation: 2.220446049250313e-16
